# IF25-40305: Sistem Teknologi Multimedia (STM)
## Modul 01 — Audio Signal Fundamentals: Resampling (Downsampling & Upsampling)

**Dosen Pengampu:** Martin C.T. Manullang, S.T., M.T., Ph.D.  
**Program Studi:** Teknik Informatika — Institut Teknologi Sumatera (ITERA)  
**Referensi Kuliah:** Slide Pertemuan 3 (*Section 1: Fondasi Audio & Resampling*)

---

### 🎯 Capaian Pembelajaran Modul
Setelah menyelesaikan modul hands-on ini, mahasiswa diharapkan mampu:
1. Menjelaskan prinsip konversi laju sampel (*Sample Rate Conversion / SRC*) dan memahami standar laju di industri multimedia/AI (8 kHz, 16 kHz, 44.1 kHz, 48 kHz).
2. Membedakan konversi laju sampel matematis vs kesalahan umum sekadar mengganti label metadata `sr`.
3. Mengimplementasikan **Downsampling (Desimasi faktor $M$)** dan membuktikan fenomena **Aliasing (*spectral fold-back*)** secara visual dan auditif.
4. Menerapkan **Filter Anti-Aliasing (Low-Pass Filter)** sebelum proses desimasi untuk menjaga integritas spektrum sinyal.
5. Mengimplementasikan **Upsampling (Interpolasi faktor $L$)** melalui dua tahapan: *Zero-Stuffing* dan *Filter Rekonstruksi (Anti-Imaging)*.
6. Menerapkan **Resampling Rasio Pecahan ($L/M$)** standar industri (contoh: audio CD 44.1 kHz ke video 48 kHz).
7. Menghindari jebakan slicing naif `audio[::2]` dan memanfaatkan pustaka standar DSP modern (`scipy.signal.resample_poly` & `librosa.resample`).

## 1. Import Library & Persiapan Lingkungan

Pustaka yang digunakan dalam modul ini meliputi:
- `numpy`: Komputasi array numerik dan manipulasi sinyal diskrit.
- `scipy.signal`: Pemrosesan sinyal digital (desain filter Butterworth, `resample_poly`, dan `chirp`).
- `matplotlib.pyplot`: Visualisasi waveform domain waktu dan spektrum domain frekuensi.
- `librosa` & `librosa.display`: Analisis audio dan visualisasi spektrogram.
- `IPython.display`: Pemutar audio interaktif langsung di dalam notebook.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import scipy.signal as signal
import librosa
import librosa.display
import IPython.display as ipd

# Konfigurasi visualisasi plot
plt.rcParams['figure.figsize'] = (10, 4)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.4
plt.rcParams['font.size'] = 10

print("Environment siap! Library berhasil dimuat.")

## 2. Kebutuhan Konversi Laju Sampel (Standar Industri)

Di industri nyata, berbagai perangkat keras, platform media, dan model kecerdasan buatan bekerja pada laju sampel yang berbeda:

| Laju Sampel ($f_s$) | Standar Penggunaan di Lapangan |
|---|---|
| **8.000 Hz (8 kHz)** | Komunikasi suara telepon seluler tradisional & walkie-talkie (rentang pita vokal ~300–3400 Hz) |
| **16.000 Hz (16 kHz)** | **Standar emas AI** (Automatic Speech Recognition: OpenAI Whisper, Siri, Google Speech) |
| **44.100 Hz (44.1 kHz)** | Audio CD standar, rekaman musik komersial, streaming Spotify |
| **48.000 Hz (48 kHz)** | Audio video profesional, bioskop, siaran TV digital, YouTube |
| **96.000 Hz (96 kHz)** | Studio rekaman audio resolusi tinggi (*mastering*) |

> ⚠️ **Peringatan Penting: Resampling Bukan Sekadar Mengganti Angka Label `sr`!**  
> Jika kita hanya mengubah label `sr` tanpa menghitung ulang titik-titik sampelnya:  
> - Memutar audio 44.1 kHz dengan label 22.05 kHz akan membuat suara diputar **setengah kecepatan dan bernada sangat rendah (suara monster)**.  
> - Memutar audio 22.05 kHz dengan label 44.1 kHz akan membuat suara diputar **dua kali lebih cepat dan melengking (efek suara kartun *chipmunk*)**.

In [ ]:
# Sintesis nada murni A4 (440 Hz) selama 2 detik pada laju standar CD (44.100 Hz)
fs_orig = 44100
duration = 2.0
t = np.linspace(0, duration, int(fs_orig * duration), endpoint=False)
f_tone = 440.0  # Frekuensi nada A4
y_orig = 0.5 * np.sin(2 * np.pi * f_tone * t)

print(f"Total sampel asli : {len(y_orig)} sampel")
print(f"Laju sampel asli  : {fs_orig} Hz")
print(f"Durasi            : {duration} detik")

# Dengarkan nada asli A4 (440 Hz)
print("Nada Asli A4 (440 Hz):")
ipd.Audio(y_orig, rate=fs_orig)

In [ ]:
# Demonstrasi kesalahan: Mengubah rate pemutar tanpa mengubah jumlah sampel data

# 1. Label diturunkan ke 22.050 Hz -> Nada melambat & pitch turun 1 oktaf (220 Hz)
print("Efek Monster (Playback rate diatur ke 22.050 Hz):")
display(ipd.Audio(y_orig, rate=22050))

# 2. Label dinaikkan ke 88.200 Hz -> Nada melesat cepat & pitch naik 1 oktaf (880 Hz)
print("Efek Chipmunk (Playback rate diatur ke 88.200 Hz):")
display(ipd.Audio(y_orig, rate=88200))

## 3. Mekanisme Downsampling (Desimasi Faktor $M$)

**Desimasi** adalah proses menurunkan laju sampel dengan faktor bilangan bulat $M$:
$$y[m] = x[M \cdot m]$$

Misalkan $M = 2$: kita menyimpan sampel dengan indeks genap ($x[0], x[2], x[4], \dots$) dan **membuang** sampel indeks ganjil ($x[1], x[3], x[5], \dots$).

### ⚠️ Bahaya Aliasing (*Spectral Fold-Back*)
Ketika laju sampel diturunkan dari $f_{s1}$ ke $f_{s2} = \frac{f_{s1}}{M}$, **batas aman Nyquist ikut turun** menjadi:
$$f_{\text{Nyquist\_baru}} = \frac{f_{s2}}{2} = \frac{f_{s1}}{2M}$$

Apabila sinyal input memiliki komponen frekuensi yang lebih tinggi dari batas Nyquist baru ($f > f_{\text{Nyquist\_baru}}$), frekuensi tersebut **tidak hilang**, melainkan **melipat balik (*spectral fold-back*) ke dalam rentang pita dasar sebagai frekuensi palsu (derau distorsi)**!

#### Contoh Perhitungan Kasus:
- Laju awal $f_{s1} = 8.000\text{ Hz}$, mengandung komponen frekuensi tinggi $f_1 = 3.500\text{ Hz}$.
- Kita lakukan downsampling dengan $M = 2 \implies f_{s2} = 4.000\text{ Hz}$.
- Batas Nyquist baru adalah $\frac{4.000}{2} = 2.000\text{ Hz}$.
- Karena $3.500\text{ Hz} > 2.000\text{ Hz}$, komponen ini melipat menjadi frekuensi alias:
  $$f_{\text{alias}} = |3.500 - 4.000| = \mathbf{500\text{ Hz}}!$$
Komponen frekuensi tinggi 3.500 Hz sekarang menyamar menjadi nada rendah 500 Hz dan mengotori sinyal utama!

Mari kita buktikan dengan simulasi spektrum FFT:

In [ ]:
# Membuat sinyal komposit: Nada rendah (500 Hz) + Nada tinggi (3500 Hz)
fs_in = 8000
t_in = np.linspace(0, 1.0, fs_in, endpoint=False)

signal_500 = 0.5 * np.sin(2 * np.pi * 500 * t_in)    # Sinyal primer
signal_3500 = 0.5 * np.sin(2 * np.pi * 3500 * t_in)  # Sinyal frekuensi tinggi
x = signal_500 + signal_3500

# Downsampling naif: memotong array dengan slicing [::2] tanpa filtering
M = 2
y_naive = x[::M]
fs_out = fs_in // M  # 4000 Hz

print(f"Sampel awal      : {len(x)} sampel pada {fs_in} Hz")
print(f"Sampel terpotong : {len(y_naive)} sampel pada {fs_out} Hz")
print(f"Batas Nyquist baru : {fs_out / 2} Hz")

In [ ]:
# Fungsi pembantu untuk memplot spektrum frekuensi dengan FFT
def plot_spectrum(sig, fs, title, ax, color='#0ea5e9'):
    N = len(sig)
    freqs = np.fft.rfftfreq(N, 1 / fs)
    magnitude = np.abs(np.fft.rfft(sig)) / (N / 2)
    ax.plot(freqs, magnitude, color=color, lw=1.6)
    ax.set_title(title, fontweight='bold', fontsize=11)
    ax.set_xlabel('Frekuensi (Hz)')
    ax.set_ylabel('Magnitudo')
    ax.set_xlim(0, fs / 2)
    ax.axvline(fs / 2, color='#ef4444', linestyle='--', alpha=0.8, label=f'Batas Nyquist ({fs/2:.0f} Hz)')
    ax.legend(loc='upper right')

fig, axes = plt.subplots(2, 1, figsize=(10, 6))
plot_spectrum(x, fs_in, "Spektrum Sinyal Asli (fs = 8000 Hz) — Puncak di 500 Hz & 3500 Hz", axes[0], color='#0284c7')
plot_spectrum(y_naive, fs_out, "Spektrum Downsampling Naif [::2] (fs = 4000 Hz) — Terjadi ALIASING di 500 Hz!", axes[1], color='#dc2626')
plt.tight_layout()
plt.show()

**Penjelasan Hasil Visualisasi:**
- Pada sinyal asli (grafik atas), komponen 500 Hz dan 3.500 Hz masing-masing memiliki magnitudo tepat **0.5**.
- Pada sinyal hasil desimasi naif (grafik bawah), komponen 3.500 Hz melipat balik tepat ke posisi **500 Hz**, sehingga nilai magnitudo di frekuensi 500 Hz melonjak menjadi **1.0** ($0.5 + 0.5$).
- Hal ini membuktikan bahaya distorsi aliasing yang merusak sinyal asli.

## 4. Solusi Standar DSP: Filter Anti-Aliasing

### 🛡️ Aturan Emas Downsampling:
> **Selalu terapkan Low-Pass Filter (LPF) SEBELUM sampel dibuang!**
>
> Frekuensi cutoff filter LPF harus memenuhi:
> $$f_{\text{cut}} \le \frac{f_{\text{target}}}{2} = \frac{f_s}{2M}$$

Mari kita bandingkan tiga pendekatan implementasi:
1. **Pendekatan Naif**: `x[::M]` (Tanpa filter $\to$ rusak karena aliasing).
2. **Pendekatan Manual**: Butterworth Low-Pass Filter $\to$ Slicing desimasi.
3. **Standar Industri**: `scipy.signal.resample_poly` dan `librosa.resample` (menggunakan arsitektur Polyphase Filter FIR yang sangat efisien dan otomatis terlindungi anti-alias).

In [ ]:
# 1. Implementasi Manual: Butterworth Low-Pass Filter
nyq_target = fs_out / 2  # 2000 Hz
cutoff = 1800  # Cutoff aman di bawah batas Nyquist target
b, a = signal.butter(N=8, Wn=cutoff / (fs_in / 2), btype='low')

# Terapkan filter ke sinyal SEBELUM membuang sampel
x_filtered = signal.filtfilt(b, a, x)
y_filtered = x_filtered[::M]

# 2. Implementasi Standar Industri: scipy.signal.resample_poly
y_poly = signal.resample_poly(x, up=1, down=M)

# 3. Implementasi Librosa
y_librosa = librosa.resample(x, orig_sr=fs_in, target_sr=fs_out)

# Visualisasi komparasi ketiga metode
fig, axes = plt.subplots(3, 1, figsize=(10, 8))
plot_spectrum(y_naive, fs_out, "1. Downsampling Naif x[::2] (RUSAK: Aliasing di 500 Hz)", axes[0], color='#dc2626')
plot_spectrum(y_filtered, fs_out, "2. Butterworth LPF + Desimasi (BERSIH: Komponen 3500 Hz teredam)", axes[1], color='#059669')
plot_spectrum(y_poly, fs_out, "3. scipy.signal.resample_poly (STANDAR DSP: Akurat & Optimal)", axes[2], color='#7c3aed')
plt.tight_layout()
plt.show()

## 5. Mekanisme Upsampling (Interpolasi Faktor $L$)

**Upsampling** adalah proses menaikkan laju sampel dengan faktor bulat $L$:
$$f_{s2} = L \cdot f_{s1}$$

Mekanisme upsampling terdiri dari **dua tahap berurutan**:

```
Sinyal Asli (fs1) 
       ↓
Tahap 1: Zero-Stuffing (Sisipkan L-1 angka nol di antara sampel asli)
       ↓  [Menghasilkan Spectral Imaging / distorsi desis tinggi]
Tahap 2: Filter Rekonstruksi (Low-Pass Filter Interpolasi dengan Gain × L)
       ↓
Sinyal Upsampled Bersih (fs2)
```

1. **Tahap 1: Zero-Stuffing (Penyisipan Nol)**  
   Sisipkan $(L-1)$ angka nol di antara setiap sampel asli.  
   Contoh ($L=3$): dari array $[x_0, x_1, x_2]$ menjadi $[x_0, \mathbf{0}, \mathbf{0}, x_1, \mathbf{0}, \mathbf{0}, x_2, \mathbf{0}, \mathbf{0}]$.  
   *Dampak:* Sinyal menjadi diskrit patah-patah dan menimbulkan duplikasi bayangan frekuensi tinggi (*spectral imaging*).

2. **Tahap 2: Filter Rekonstruksi (Anti-Imaging)**  
   Filter LPF dengan batas cutoff $f_{\text{cut}} \le \frac{f_{s1}}{2}$ bertugas menghaluskan kurva (*connect-the-dots*) dan menginterpolasi nilai di antara sampel asli. Filter diberi penguatan (*gain*) sebesar $L$ agar energi sinyal tetap konsisten.

In [ ]:
# Demonstrasi 3 Tahap Upsampling
L = 3  # Menaikkan laju sampel 3x lipat
fs1 = 4000
t1 = np.linspace(0, 0.01, int(fs1 * 0.01), endpoint=False)
f_wave = 300  # Gelombang sinus 300 Hz
x_up = np.sin(2 * np.pi * f_wave * t1)

# Tahap 1: Zero-Stuffing (Sisipkan L-1 = 2 angka nol)
fs2 = fs1 * L  # 12.000 Hz
x_zero_stuffed = np.zeros(len(x_up) * L)
x_zero_stuffed[::L] = x_up

# Tahap 2: Filter Rekonstruksi (LPF Anti-Imaging) dengan Gain = L
nyq_new = fs2 / 2
cutoff_up = (fs1 / 2) * 0.9  # Cutoff sedikit di bawah batas Nyquist awal
b_up, a_up = signal.butter(N=6, Wn=cutoff_up / nyq_new, btype='low')
x_interpolated = L * signal.filtfilt(b_up, a_up, x_zero_stuffed)

# Visualisasi ketiga tahap upsampling
t_dense = np.linspace(0, 0.01, len(x_zero_stuffed), endpoint=False)

fig, axes = plt.subplots(3, 1, figsize=(10, 7))
axes[0].stem(t1 * 1000, x_up, linefmt='C0-', markerfmt='C0o', basefmt='k-')
axes[0].set_title("Tahap 1: Sinyal Asli (fs = 4000 Hz, 40 sampel)", fontweight='bold')
axes[0].set_ylabel("Amplitudo")

axes[1].stem(t_dense * 1000, x_zero_stuffed, linefmt='C3--', markerfmt='C3^', basefmt='k-')
axes[1].set_title("Tahap 2: Zero-Stuffing (L = 3, 120 sampel, disisipkan 2 nol antar sampel)", fontweight='bold')
axes[1].set_ylabel("Amplitudo")

axes[2].plot(t_dense * 1000, x_interpolated, color='#059669', lw=2, label='Hasil Interpolasi LPF')
axes[2].stem(t1 * 1000, x_up, linefmt='C0:', markerfmt='C0o', basefmt='k-', label='Sampel Asli')
axes[2].set_title("Tahap 3: Sinyal Terinterpolasi Mulus (Kurva Halus Menghubungkan Titik)", fontweight='bold')
axes[2].set_xlabel("Waktu (milidetik)")
axes[2].set_ylabel("Amplitudo")
axes[2].legend(loc='upper right')

plt.tight_layout()
plt.show()

## 6. Resampling Rasio Pecahan ($L/M$) — Kasus Nyata 44.1 kHz ke 48 kHz

Di dunia broadcasting dan video editing, konversi antara audio CD (44.1 kHz) dan video/YouTube (48 kHz) sangat umum terjadi.
Karena $48.000$ bukan kelipatan bulat dari $44.100$, rasionya disederhanakan menjadi pecahan:
$$\frac{f_{s2}}{f_{s1}} = \frac{48.000}{44.100} = \frac{160}{147}$$

Rantai konversi standar industri:
1. **Upsample faktor $L = 160$** (sisipkan 159 nol).
2. **Satu Filter LPF Tunggal** dengan cutoff $\approx 22.050\text{ Hz}$ (berfungsi ganda: anti-imaging untuk $L=160$ sekaligus anti-aliasing untuk $M=147$).
3. **Downsample faktor $M = 147$** (ambil 1 sampel tiap 147).

Dengan fungsi `scipy.signal.resample_poly`, komputasi ini berjalan instan menggunakan Polyphase Filtering:

In [ ]:
# Kasus Nyata: Konversi CD 44.1 kHz ke Video 48 kHz
fs_cd = 44100
fs_video = 48000

# Sinyal uji 1 detik (dua nada harmonik: 440 Hz dan 1000 Hz)
t_cd = np.linspace(0, 1.0, fs_cd, endpoint=False)
y_cd = 0.4 * np.sin(2 * np.pi * 440 * t_cd) + 0.3 * np.sin(2 * np.pi * 1000 * t_cd)

# Rasio pecahan paling sederhana: 160 / 147
L_frac = 160
M_frac = 147

# 1. Menggunakan scipy.signal.resample_poly
y_video_poly = signal.resample_poly(y_cd, up=L_frac, down=M_frac)

# 2. Menggunakan librosa.resample
y_video_librosa = librosa.resample(y_cd, orig_sr=fs_cd, target_sr=fs_video)

print(f"Sampel awal (44.100 Hz)   : {len(y_cd)} sampel")
print(f"Sampel target teoritis    : {int(len(y_cd) * 48000 / 44100)} sampel")
print(f"Hasil resample_poly       : {len(y_video_poly)} sampel")
print(f"Hasil librosa.resample    : {len(y_video_librosa)} sampel")

## 7. Demonstrasi Audio Nyata: Chirp Signal (Sapuan Frekuensi)

Untuk mengamati dan **mendengar** fenomena aliasing secara dramatis, kita membuat **Chirp Signal** (sinyal yang frekuensinya terus merangkak naik dari $100\text{ Hz}$ sampai $3.800\text{ Hz}$ pada $f_s = 8.000\text{ Hz}$).

Jika sinyal ini kita downsample $4\times$ ($M = 4$) ke $f_{s2} = 2.000\text{ Hz}$ (batas Nyquist baru $= 1.000\text{ Hz}$):
- **Downsample Naif**: Frekuensi yang melebihi 1.000 Hz akan melipat memantul ke bawah, menghasilkan pola spektrogram berbentuk huruf **V** dan suara aneh yang naik-lalu-turun!
- **Downsample DSP**: Frekuensi di atas 1.000 Hz diredam bersih, suara berhenti saat menyentuh batas Nyquist.

In [ ]:
# Sintesis sinyal chirp 3 detik (100 Hz s.d. 3800 Hz pada fs = 8000 Hz)
fs_chirp = 8000
dur_chirp = 3.0
t_chirp = np.linspace(0, dur_chirp, int(fs_chirp * dur_chirp), endpoint=False)
y_chirp = signal.chirp(t_chirp, f0=100, t1=dur_chirp, f1=3800, method='linear')

# Downsample 4x (Laju baru = 2000 Hz, Batas Nyquist baru = 1000 Hz)
M_chirp = 4
fs_target_chirp = fs_chirp // M_chirp  # 2000 Hz

# Metode 1: Naif (slicing [::4])
y_chirp_naive = y_chirp[::M_chirp]

# Metode 2: DSP Polyphase Filter
y_chirp_clean = signal.resample_poly(y_chirp, up=1, down=M_chirp)

# Visualisasi Spektrogram Perbandingan
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

D_naive = np.abs(librosa.stft(y_chirp_naive, n_fft=256, hop_length=64))
librosa.display.specshow(librosa.amplitude_to_db(D_naive, ref=np.max), sr=fs_target_chirp, 
                         x_axis='time', y_axis='hz', ax=axes[0], cmap='magma')
axes[0].set_title("1. Downsampling Naif [::4] (Pola 'V' Akibat Aliasing)", fontweight='bold')
axes[0].set_ylim(0, 1000)

D_clean = np.abs(librosa.stft(y_chirp_clean, n_fft=256, hop_length=64))
librosa.display.specshow(librosa.amplitude_to_db(D_clean, ref=np.max), sr=fs_target_chirp, 
                         x_axis='time', y_axis='hz', ax=axes[1], cmap='magma')
axes[1].set_title("2. Downsampling Standar DSP (Bersih Tanpa Pantulan)", fontweight='bold')
axes[1].set_ylim(0, 1000)

plt.tight_layout()
plt.show()

In [ ]:
# Dengarkan perbedaannya secara auditif:
print("1. Audio Chirp Asli (8000 Hz) — Frekuensi naik konsisten dari 100 Hz ke 3800 Hz:")
display(ipd.Audio(y_chirp, rate=fs_chirp))

print("2. Downsample Naif (2000 Hz) — Dengarkan nada memantul turun kembali saat lewat 1000 Hz:")
display(ipd.Audio(y_chirp_naive, rate=fs_target_chirp))

print("3. Downsample DSP (2000 Hz) — Nada berhenti dengan bersih saat menyentuh batas 1000 Hz:")
display(ipd.Audio(y_chirp_clean, rate=fs_target_chirp))

## 8. Latihan Terpandu Mahasiswa (Sesuai Soal Slide Pertemuan 3)

Kerjakan latihan-latihan berikut untuk memperkuat pemahaman matematis dan implementasi kode Anda.

---

### 📝 Latihan 1: Desimasi Sinyal ($M = 2$)
Diberikan sebuah array sinyal sensor input berdurasi $T = 1.0\text{ detik}$ berisi 20 elemen sampel:
```python
x1 = np.array([0.0, 0.6, 1.0, 1.0, 0.6, 0.0, -0.6, -1.0, -1.0, -0.6,
               0.0, 0.6, 1.0, 1.0, 0.6, 0.0, -0.6, -1.0, -1.0, -0.6])
```

**Instruksi:**
1. Hitung laju sampel awal ($f_{s1}$) dan faktor desimasi $M$ jika target laju adalah $f_{s2} = 10\text{ Hz}$.
2. Hitung batas Nyquist baru serta tentukan batas frekuensi cutoff filter LPF yang disyaratkan.
3. Tuliskan array hasil output $y_1[m]$ (harus tepat 10 sampel).

In [ ]:
# --- PENYELESAIAN LATIHAN 1 ---
x1 = np.array([0.0, 0.6, 1.0, 1.0, 0.6, 0.0, -0.6, -1.0, -1.0, -0.6,
               0.0, 0.6, 1.0, 1.0, 0.6, 0.0, -0.6, -1.0, -1.0, -0.6])

# 1. Perhitungan fs1 dan faktor M
fs1_lat1 = len(x1) / 1.0  # 20 Hz
fs2_lat1 = 10.0           # 10 Hz
M1 = int(fs1_lat1 / fs2_lat1)  # 2

# 2. Batas Nyquist baru
nyquist_baru1 = fs2_lat1 / 2.0  # 5 Hz

# 3. Hasil desimasi (ambil indeks genap n = 0, 2, 4, ...)
y1 = x1[::M1]

print(f"Laju sampel awal (fs1)  : {fs1_lat1} Hz")
print(f"Faktor desimasi (M)      : {M1}")
print(f"Batas Nyquist baru       : {nyquist_baru1} Hz (LPF cutoff <= 5 Hz)")
print(f"Array output (10 sampel) : {y1}")

---

### 📝 Latihan 2: Interpolasi Sinyal ($L = 3$)
Diberikan sinyal audio berdurasi $T = 1.0\text{ detik}$ memiliki 6 sampel diskrit:
```python
x2 = np.array([0.0, 1.0, 0.5, -0.5, -1.0, 0.0])
```
Target konversi laju adalah $f_{s2} = 18\text{ Hz}$.

**Instruksi:**
1. Hitung laju awal ($f_{s1}$) dan tentukan faktor upsampling $L$.
2. Lakukan proses **Zero-Stuffing** (menyisipkan $L-1$ buah nol di antara setiap sampel asli).
3. Tuliskan array hasil zero-stuffing dan verifikasi jumlah total sampelnya (harus tepat 18 sampel).
4. Jelaskan peran filter LPF rekonstruksi setelah tahap penyisipan nol.

In [ ]:
# --- PENYELESAIAN LATIHAN 2 ---
x2 = np.array([0.0, 1.0, 0.5, -0.5, -1.0, 0.0])

# 1. Laju awal dan faktor L
fs1_lat2 = len(x2) / 1.0  # 6 Hz
fs2_lat2 = 18.0           # 18 Hz
L2 = int(fs2_lat2 / fs1_lat2)  # 3

# 2. Zero-stuffing (sisipkan L-1 = 2 angka nol)
x_zero_stuffed = np.zeros(len(x2) * L2)
x_zero_stuffed[::L2] = x2

print(f"Laju sampel awal (fs1)  : {fs1_lat2} Hz")
print(f"Faktor upsampling (L)    : {L2}")
print(f"Total sampel baru        : {len(x_zero_stuffed)} sampel")
print("Array Zero-Stuffing:")
print(x_zero_stuffed)

---

### 📝 Latihan 3: Analisis Risiko Aliasing Desimasi 30 Hz ke 10 Hz ($M = 3$)
Diberikan array getaran sensor 30 elemen dalam $1.0\text{ detik}$ ($f_{s1} = 30\text{ Hz}$):
```python
x3 = np.array([0.0, 0.9, 0.6, 0.5, 1.2, 1.3, 0.3, -0.2, 0.2, -0.3, -1.3, -1.2, -0.5, -0.6, -0.9,
               0.0, 0.9, 0.6, 0.5, 1.2, 1.3, 0.3, -0.2, 0.2, -0.3, -1.3, -1.2, -0.5, -0.6, -0.9])
```

Jika sinyal ini didownsample menjadi $10\text{ sampel}$ ($f_{s2} = 10\text{ Hz}$):
1. Hitung faktor desimasi $M$ dan batas Nyquist baru ($f_{\text{Nyq2}}$).
2. Apabila terdapat derau frekuensi tinggi sebesar **8 Hz**, apakah derau tersebut mengalami aliasing? Ke frekuensi berapakah derau tersebut melipat?
3. Lakukan desimasi dan tampilkan array output 10 sampel.

In [ ]:
# --- PENYELESAIAN LATIHAN 3 ---
x3 = np.array([0.0, 0.9, 0.6, 0.5, 1.2, 1.3, 0.3, -0.2, 0.2, -0.3, -1.3, -1.2, -0.5, -0.6, -0.9,
               0.0, 0.9, 0.6, 0.5, 1.2, 1.3, 0.3, -0.2, 0.2, -0.3, -1.3, -1.2, -0.5, -0.6, -0.9])

# 1. Parameter
fs1_lat3 = 30.0  # Hz
fs2_lat3 = 10.0  # Hz
M3 = int(fs1_lat3 / fs2_lat3)  # 3
nyq2_lat3 = fs2_lat3 / 2.0     # 5 Hz

# 2. Analisis Derau 8 Hz
f_noise = 8.0  # Hz
f_alias_noise = abs(f_noise - fs2_lat3)  # |8 - 10| = 2 Hz

# 3. Hasil Desimasi (ambil kelipatan 3: n = 0, 3, 6, ...)
y3 = x3[::M3]

print(f"Faktor desimasi (M)      : {M3}")
print(f"Batas Nyquist baru       : {nyq2_lat3} Hz")
print(f"Derau {f_noise} Hz > {nyq2_lat3} Hz -> MENGALAMI ALIASING!")
print(f"Derau melipat balik ke   : {f_alias_noise} Hz")
print(f"Array output (10 sampel) : {y3}")

## 9. Rangkuman & Kesimpulan

1. **Sample Rate Conversion (SRC)** bukan sekadar mengganti label metadata pada file audio. Mengubah laju sampel membutuhkan rekonstruksi titik-titik sampel secara matematis.
2. **Downsampling (Desimasi $M$)** membuang $(M-1)$ sampel untuk efisiensi memori dan komputasi model AI. Syarat mutlaknya adalah menerapkan **Filter Anti-Aliasing (LPF)** sebelum desimasi untuk mencegah *spectral fold-back*.
3. **Upsampling (Interpolasi $L$)** menaikkan resolusi waktu sinyal dengan menyisipkan $(L-1)$ angka nol (*Zero-Stuffing*), lalu menghaluskan transisi kurva menggunakan **Filter Rekonstruksi (Anti-Imaging)** dengan gain $L$.
4. **Resampling Rasio Pecahan ($L/M$)** menggabungkan upsampling dan downsampling dengan satu filter LPF tunggal di tengah.
5. Selalu gunakan pustaka standar industri seperti `scipy.signal.resample_poly` atau `librosa.resample` untuk hasil yang optimal dan bebas distorsi.

---
© IF25-40305 Sistem Teknologi Multimedia — Institut Teknologi Sumatera (ITERA).